## 1. Setup

In [1]:
import os, sys, time
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

from pyspark.sql import SparkSession, functions as F, Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.regression import (LinearRegression, RandomForestRegressor,
                                   GBTRegressor)
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing data/raw")

PROJECT = find_project_root(Path.cwd())
FIGURES = PROJECT / "docs" / "figures"
MODELS  = PROJECT / "models"
FIGURES.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.bbox"] = "tight"

def save(fig, n, name):
    p = FIGURES / f"fig{n:02d}_{name}.png"
    fig.savefig(p); plt.close(fig); print(f"  saved {p.name}")

spark = (SparkSession.builder
         .appName("ST5011CEM_Modelling")
         .master("local[*]")
         .config("spark.driver.memory", "8g")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.sql.adaptive.enabled", "false")
         .config("spark.sql.session.timeZone", "Europe/London")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark UI:", spark.sparkContext.uiWebUrl)

Spark UI: http://ujwal:4040


In [2]:
delays = spark.read.parquet((PROJECT / "data" / "processed" / "observed_delays").as_posix())
print(f"Observations: {delays.count():,}")
print(f"Partitions  : {delays.rdd.getNumPartitions()}")

Observations: 1,201,509
Partitions  : 20


## 2. Feature engineering

### 2.1 Temporal features

In [3]:
df = (delays
      .withColumn("hour",        F.hour("recorded_ts"))
      .withColumn("minute",      F.minute("recorded_ts"))
      .withColumn("dow_n",       F.dayofweek("recorded_ts"))
      .withColumn("is_weekend",  F.dayofweek("recorded_ts").isin([1, 7]).cast("int"))
      .withColumn("is_am_peak",  F.hour("recorded_ts").between(7, 9).cast("int"))
      .withColumn("is_pm_peak",  F.hour("recorded_ts").between(16, 18).cast("int"))
      .withColumn("mins_since_midnight",
                  F.hour("recorded_ts") * 60 + F.minute("recorded_ts"))
      # cyclical encoding so 23:00 and 00:00 are near each other
      .withColumn("hour_sin", F.sin(2 * np.pi * F.col("hour") / 24))
      .withColumn("hour_cos", F.cos(2 * np.pi * F.col("hour") / 24)))

print("Temporal features added.")
df.select("recorded_ts", "hour", "dow_n", "is_weekend",
          "is_am_peak", "is_pm_peak").show(5)

Temporal features added.
+-------------------+----+-----+----------+----------+----------+
|        recorded_ts|hour|dow_n|is_weekend|is_am_peak|is_pm_peak|
+-------------------+----+-----+----------+----------+----------+
|2026-07-24 18:30:41|  18|    6|         0|         0|         1|
|2026-07-25 14:28:56|  14|    7|         1|         0|         0|
|2026-07-25 16:32:32|  16|    7|         1|         0|         1|
|2026-07-24 14:31:03|  14|    6|         0|         0|         0|
|2026-07-25 14:30:19|  14|    7|         1|         0|         0|
+-------------------+----+-----+----------+----------+----------+
only showing top 5 rows



### 2.2 Lag features

A window partitioned by vehicle and date, ordered by time, gives each
observation access to the same bus's previous stops. `prev_delay` is the delay
one stop earlier, `prev_delay_2` two stops earlier, and `delay_trend` the
difference between them — whether the bus is losing or recovering time.

Rows without a predecessor (the first observation for each vehicle each day) are
dropped, since the model cannot be given a value that would not exist in
production.

In [4]:
w_vehicle = Window.partitionBy("vehicle_ref", "obs_date", "trip_id") \
                  .orderBy("stop_sequence")

df = (df
      .withColumn("prev_delay",   F.lag("delay_min", 1).over(w_vehicle))
      .withColumn("prev_delay_2", F.lag("delay_min", 2).over(w_vehicle))
      .withColumn("prev_ts",      F.lag("recorded_ts", 1).over(w_vehicle))
      .withColumn("prev_seq",     F.lag("stop_sequence", 1).over(w_vehicle)))

df = (df
      .withColumn("delay_trend", F.col("prev_delay") - F.col("prev_delay_2"))
      .withColumn("secs_since_prev",
                  F.col("recorded_ts").cast("long") - F.col("prev_ts").cast("long"))
      .withColumn("stops_since_prev", F.col("stop_sequence") - F.col("prev_seq")))

before = df.count()
df = df.filter(
    F.col("prev_delay").isNotNull() & F.col("prev_delay_2").isNotNull() &
    F.col("stops_since_prev").between(1, 3) &
    (F.col("dist_m") <= 60) &
    (F.col("secs_since_prev") >= 30)
)
after = df.count()

print(f"Before lag filtering : {before:,}")
print(f"After                : {after:,}   ({100*after/before:.1f}% retained)")
print("\nCorrelation of prev_delay with delay_min: "
      f"{df.stat.corr('prev_delay', 'delay_min'):.4f}")

df = (df
      .withColumn("prev_arrival_sec", F.lag("arrival_sec", 1).over(w_vehicle))
      .withColumn("scheduled_gap", F.col("arrival_sec") - F.col("prev_arrival_sec")))

Before lag filtering : 1,201,509
After                : 442,573   (36.8% retained)

Correlation of prev_delay with delay_min: 0.8097


### 2.3 Temporal train / test split

A random split would let the model see future observations while predicting past
ones, inflating every metric. The data is therefore split chronologically: the
earliest 75% trains, the most recent 25% tests. This mirrors how the system would
actually be deployed — trained on history, applied to what comes next.

In [5]:
cut = df.approxQuantile("obs_sec", [0.75], 0.001)[0]
bounds = df.select(F.min("recorded_ts").alias("lo"), F.max("recorded_ts").alias("hi")).collect()[0]
span = (bounds.hi - bounds.lo).total_seconds()
split_ts = bounds.lo + pd.Timedelta(seconds=span * 0.75)

train = df.filter(F.col("recorded_ts") <  F.lit(split_ts))
test  = df.filter(F.col("recorded_ts") >= F.lit(split_ts))

n_tr, n_te = train.count(), test.count()
print(f"Data spans   : {bounds.lo}  ->  {bounds.hi}")
print(f"Split at     : {split_ts}")
print(f"Training rows: {n_tr:,}  ({100*n_tr/(n_tr+n_te):.1f}%)")
print(f"Test rows    : {n_te:,}  ({100*n_te/(n_tr+n_te):.1f}%)")

Data spans   : 2026-07-24 14:54:23  ->  2026-07-27 12:55:54
Split at     : 2026-07-26 19:25:31.250000
Training rows: 406,068  (91.8%)
Test rows    : 36,505  (8.2%)


### 2.4 Historical aggregates (target encoding)

Some stops and routes are systematically worse than others. Their historical mean
delay is a strong feature, but it must be computed on the **training set only** —
using test data to build it would leak the answer.

In [6]:
stop_hist = (train.groupBy("stop_id")
             .agg(F.avg("delay_min").alias("stop_hist_delay"),
                  F.count("*").alias("stop_hist_n")))

route_hist = (train.groupBy("route_short_name")
              .agg(F.avg("delay_min").alias("route_hist_delay"),
                   F.count("*").alias("route_hist_n")))

global_mean = train.select(F.avg("delay_min")).collect()[0][0]
print(f"Training-set global mean delay: {global_mean:.3f} min")

def attach(d):
    return (d.join(F.broadcast(stop_hist),  "stop_id",          "left")
             .join(F.broadcast(route_hist), "route_short_name", "left")
             .fillna({"stop_hist_delay":  global_mean, "stop_hist_n":  0,
                      "route_hist_delay": global_mean, "route_hist_n": 0}))

train = attach(train).cache()
test  = attach(test).cache()
print(f"Train {train.count():,} | Test {test.count():,}")

Training-set global mean delay: 1.167 min
Train 406,068 | Test 36,505


### 2.5 Assembling the feature vector

`agency_name` and `direction_ref` are categorical, so they are indexed. Tree
ensembles handle indexed categories directly; one-hot encoding would add many
sparse columns for no gain here.

In [7]:
FEATURES = [
    # autoregressive - known at prediction time
    "prev_delay", "prev_delay_2", "delay_trend",
    # journey context from the timetable
    "stop_sequence", "stops_since_prev", "scheduled_gap",
    # historical behaviour
    "stop_hist_delay", "route_hist_delay",
    # temporal
    "hour_sin", "hour_cos", "mins_since_midnight",
    "is_weekend", "is_am_peak", "is_pm_peak", "dow_n",
    # spatial
    "stop_lat", "stop_lon",
    # categorical
    "agency_idx", "direction_idx",
]
indexers = [
    StringIndexer(inputCol="agency_name",   outputCol="agency_idx",
                  handleInvalid="keep"),
    StringIndexer(inputCol="direction_ref", outputCol="direction_idx",
                  handleInvalid="keep"),
]
assembler = VectorAssembler(inputCols=FEATURES, outputCol="features",
                            handleInvalid="skip")

prep = Pipeline(stages=indexers + [assembler]).fit(train)
train_v = prep.transform(train).select("features", F.col("delay_min").alias("label"),
                                       "recorded_ts", "agency_name")
test_v  = prep.transform(test).select("features", F.col("delay_min").alias("label"),
                                      "recorded_ts", "agency_name")
train_v.cache(); test_v.cache()

print(f"Feature vector length: {len(FEATURES)}")
print(f"Train {train_v.count():,} | Test {test_v.count():,}")
train_v.select("features", "label").show(3, truncate=80)

Feature vector length: 19
Train 369,239 | Test 32,392
+--------------------------------------------------------------------------------+-------------------+
|                                                                        features|              label|
+--------------------------------------------------------------------------------+-------------------+
|[-2.433333333333333,-1.9833333333333334,-0.44999999999999973,8.0,1.0,240.0,0....|               -2.9|
|[-1.4666666666666666,-2.9,1.4333333333333333,22.0,1.0,840.0,0.370379537953795...|-1.4333333333333333|
|[-1.4333333333333333,-1.4666666666666666,0.033333333333333215,23.0,1.0,60.0,1...|0.03333333333333333|
+--------------------------------------------------------------------------------+-------------------+
only showing top 3 rows



## 3. Model comparison

Three algorithms with different complexity profiles:

| Model | Training complexity | Notes |
|---|---|---|
| Linear Regression | O(n d^2 + d^3) | Baseline; assumes linear additive effects |
| Random Forest | O(k n log n * d) | Bagged trees, parallel, resists overfitting |
| Gradient-Boosted Trees | O(k n log n * d) | Sequential boosting, usually most accurate, slowest |

where n = rows, d = features, k = trees.

The brief also asks for **Model Efficiency**: accuracy achieved relative to
training cost. Training time is recorded for each model and RMSE per second
reported alongside the accuracy metrics.

In [8]:
evaluators = {
    "rmse": RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse"),
    "mae":  RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="mae"),
    "r2":   RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2"),
}

results = []
fitted  = {}

def run(name, estimator):
    print(f"\n--- {name} ---")
    t0 = time.time()
    model = estimator.fit(train_v)
    train_secs = time.time() - t0

    t0 = time.time()
    preds = model.transform(test_v).cache()
    preds.count()
    pred_secs = time.time() - t0

    row = {"model": name, "train_secs": round(train_secs, 2),
           "predict_secs": round(pred_secs, 2)}
    for k, ev in evaluators.items():
        row[k] = round(ev.evaluate(preds), 4)
    row["rmse_per_train_sec"] = round(row["rmse"] / max(train_secs, 1e-9), 4)

    results.append(row)
    fitted[name] = (model, preds)
    print(f"RMSE {row['rmse']:.4f} | MAE {row['mae']:.4f} | R2 {row['r2']:.4f} "
          f"| trained in {train_secs:.1f}s")
    return model

In [9]:
lr_model = run("Linear Regression",
               LinearRegression(featuresCol="features", labelCol="label",
                                maxIter=50, regParam=0.01, elasticNetParam=0.0))


--- Linear Regression ---
RMSE 1.6539 | MAE 1.0660 | R2 0.6057 | trained in 1.8s


In [10]:
rf_model = run("Random Forest",
               RandomForestRegressor(featuresCol="features", labelCol="label",
                                     numTrees=60, maxDepth=10, seed=42,
                                     subsamplingRate=0.8))


--- Random Forest ---
RMSE 1.4598 | MAE 0.7335 | R2 0.6928 | trained in 25.3s


In [11]:
gbt_model = run("Gradient-Boosted Trees",
                GBTRegressor(featuresCol="features", labelCol="label",
                             maxIter=60, maxDepth=6, stepSize=0.1, seed=42))


--- Gradient-Boosted Trees ---
RMSE 5.3644 | MAE 3.1916 | R2 -3.1481 | trained in 42.9s


In [12]:
comparison = pd.DataFrame(results)[
    ["model", "rmse", "mae", "r2", "train_secs", "predict_secs", "rmse_per_train_sec"]
].sort_values("rmse")
print("Model comparison on held-out test set:\n")
print(comparison.to_string(index=False))

best_name = comparison.iloc[0]["model"]
print(f"\nBest by RMSE: {best_name}")
comparison

Model comparison on held-out test set:

                 model   rmse    mae      r2  train_secs  predict_secs  rmse_per_train_sec
         Random Forest 1.4598 0.7335  0.6928       25.34          0.91              0.0576
     Linear Regression 1.6539 1.0660  0.6057        1.83          0.41              0.9032
Gradient-Boosted Trees 5.3644 3.1916 -3.1481       42.92          0.34              0.1250

Best by RMSE: Random Forest


,model,rmse,mae,r2,train_secs,predict_secs,rmse_per_train_sec
1,Random Forest,1.4598,0.7335,0.6928,25.34,0.91,0.0576
0,Linear Regression,1.6539,1.0660,0.6057,1.83,0.41,0.9032
2,Gradient-Boosted Trees,5.3644,3.1916,-3.1481,42.92,0.34,0.1250


In [13]:
# Naive persistence baseline: assume delay is unchanged from the previous stop
baseline = (test
            .withColumn("prediction", F.col("prev_delay"))
            .select(F.col("delay_min").alias("label"), "prediction"))

print("Persistence baseline (predict delay = prev_delay):")
base_row = {"model": "Persistence baseline"}
for k, ev in evaluators.items():
    base_row[k] = round(ev.evaluate(baseline), 4)
    print(f"  {k.upper():<5} {base_row[k]:.4f}")

best_rmse = comparison["rmse"].min()
gain = 100 * (base_row["rmse"] - best_rmse) / base_row["rmse"]
print(f"\nBest model improves on baseline by {gain:.1f}% RMSE")

Persistence baseline (predict delay = prev_delay):
  RMSE  1.7512
  MAE   0.7346
  R2    0.5625

Best model improves on baseline by 16.6% RMSE


In [14]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor

w_ahead = Window.partitionBy("vehicle_ref", "obs_date", "trip_id") \
                .orderBy("stop_sequence")

# Give the model the SAME information the baseline uses: the current delay.
HZ_FEATURES = ["delay_min"] + FEATURES
hz_asm = VectorAssembler(inputCols=HZ_FEATURES, outputCol="hz_features",
                         handleInvalid="skip")

horizon_results = []

for k in [1, 3, 5, 10]:
    def prep_h(d):
        y = prep.transform(d).withColumn("target", F.lead("delay_min", k).over(w_ahead))
        return y.filter(F.col("target").isNotNull())

    tr_h, te_h = prep_h(train), prep_h(test)
    if te_h.count() < 500:
        print(f"horizon {k}: too few rows, skipping"); continue

    tr_v = hz_asm.transform(tr_h).select("hz_features", F.col("target").alias("label"))
    te_v = hz_asm.transform(te_h).select("hz_features", F.col("target").alias("label"),
                                         "delay_min")

    rf_h = RandomForestRegressor(featuresCol="hz_features", labelCol="label",
                                 numTrees=60, maxDepth=10, seed=42).fit(tr_v)
    pr = rf_h.transform(te_v)

    base = te_v.withColumn("prediction", F.col("delay_min")).select("label", "prediction")

    m_rmse, b_rmse = evaluators["rmse"].evaluate(pr), evaluators["rmse"].evaluate(base)
    m_mae,  b_mae  = evaluators["mae"].evaluate(pr),  evaluators["mae"].evaluate(base)

    horizon_results.append({
        "stops_ahead": k,
        "model_rmse": round(m_rmse, 3), "baseline_rmse": round(b_rmse, 3),
        "model_mae":  round(m_mae, 3),  "baseline_mae":  round(b_mae, 3),
        "rmse_gain_pct": round(100*(b_rmse-m_rmse)/b_rmse, 1),
        "mae_gain_pct":  round(100*(b_mae-m_mae)/b_mae, 1),
    })
    print(f"horizon {k:>2} stops | model RMSE {m_rmse:.3f} vs baseline {b_rmse:.3f} "
          f"| gain {horizon_results[-1]['rmse_gain_pct']:+.1f}%")

hz = pd.DataFrame(horizon_results)
print("\n", hz.to_string(index=False))

horizon  1 stops | model RMSE 1.584 vs baseline 1.465 | gain -8.1%
horizon  3 stops | model RMSE 1.946 vs baseline 1.869 | gain -4.1%
horizon  5 stops | model RMSE 2.151 vs baseline 2.105 | gain -2.2%
horizon 10 stops | model RMSE 2.545 vs baseline 2.439 | gain -4.3%

  stops_ahead  model_rmse  baseline_rmse  model_mae  baseline_mae  rmse_gain_pct  mae_gain_pct
           1       1.584          1.465      0.850         0.743           -8.1         -14.3
           3       1.946          1.869      1.231         1.180           -4.1          -4.3
           5       2.151          2.105      1.415         1.415           -2.2           0.0
          10       2.545          2.439      1.705         1.687           -4.3          -1.1


In [15]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(hz["stops_ahead"], hz["baseline_rmse"], marker="s", label="Persistence baseline", color="#e76f51")
ax.plot(hz["stops_ahead"], hz["model_rmse"], marker="o", label="Random Forest", color="#2a9d8f")
ax.set_xlabel("Prediction horizon (stops ahead)")
ax.set_ylabel("RMSE (minutes)")
ax.set_title("Model Value Against Naive Persistence by Prediction Horizon")
ax.legend()
save(fig, 13, "horizon_comparison")

  saved fig13_horizon_comparison.png


## 4. Cross-validated hyperparameter tuning

`CrossValidator` on the best-performing family, as required by the brief's
instruction to document the ML pipeline including CrossValidator. Folds are drawn
from the training partition only, so the test set stays untouched.

In [16]:
rf = RandomForestRegressor(featuresCol="features", labelCol="label", seed=42)

grid = (ParamGridBuilder()
        .addGrid(rf.numTrees, [40, 80])
        .addGrid(rf.maxDepth, [8, 12])
        .build())

cv = CrossValidator(estimator=rf,
                    estimatorParamMaps=grid,
                    evaluator=evaluators["rmse"],
                    numFolds=3,
                    parallelism=1,          # was 2 - halves peak memory
                    seed=42)

# Tune on a sample; the full set isn't needed to pick hyperparameters
cv_train = train_v.sample(fraction=0.3, seed=42).cache()
print(f"Tuning on {cv_train.count():,} sampled rows")

t0 = time.time()
cv_model = cv.fit(cv_train)
cv_secs = time.time() - t0

best = cv_model.bestModel
print(f"Cross-validation completed in {cv_secs:.1f}s over {len(grid)} configurations")
print(f"Best numTrees : {best.getNumTrees}")
print(f"Best maxDepth : {best.getMaxDepth()}")

cv_preds = best.transform(test_v).cache()
print("\nTuned Random Forest on test set:")
for k, ev in evaluators.items():
    print(f"  {k.upper():<5} {ev.evaluate(cv_preds):.4f}")

Tuning on 110,871 sampled rows
Cross-validation completed in 189.3s over 4 configurations
Best numTrees : 80
Best maxDepth : 12

Tuned Random Forest on test set:
  RMSE  1.5279
  MAE   0.7589
  R2    0.6635


In [17]:
avg = cv_model.avgMetrics
print("Mean cross-validated RMSE per configuration:")
for params, score in zip(grid, avg):
    desc = ", ".join(f"{k.name}={v}" for k, v in params.items())
    print(f"  {desc:<32} RMSE {score:.4f}")

Mean cross-validated RMSE per configuration:
  numTrees=40, maxDepth=8          RMSE 1.6887
  numTrees=40, maxDepth=12         RMSE 1.6334
  numTrees=80, maxDepth=8          RMSE 1.6822
  numTrees=80, maxDepth=12         RMSE 1.6311


## 5. Figure 9 — Model comparison

In [18]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, metric, title, lower_better in [
        (axes[0], "rmse", "RMSE (minutes)", True),
        (axes[1], "mae",  "MAE (minutes)",  True),
        (axes[2], "r2",   "R-squared",      False)]:
    d = comparison.sort_values(metric, ascending=lower_better)
    bars = ax.bar(range(len(d)), d[metric],
                  color=["#2a9d8f", "#e9c46a", "#e76f51"][:len(d)])
    ax.set_xticks(range(len(d)))
    ax.set_xticklabels([m.replace(" ", "\n") for m in d["model"]], fontsize=8)
    ax.set_title(title)
    for b, v in zip(bars, d[metric]):
        ax.text(b.get_x() + b.get_width()/2, b.get_height(), f"{v:.3f}",
                ha="center", va="bottom", fontsize=8)
fig.suptitle("Regression Model Comparison on Held-out Test Set")
save(fig, 9, "model_comparison")

  saved fig09_model_comparison.png


## 6. Figure 10 — Feature importance

In [19]:
imp = pd.DataFrame({"feature": FEATURES,
                    "importance": best.featureImportances.toArray()}) \
        .sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(8, 6.5))
ax.barh(imp["feature"], imp["importance"], color="#3b7dd8")
ax.set_xlabel("Gini importance")
ax.set_title("Feature Importance — Tuned Random Forest")
save(fig, 10, "feature_importance")

print("Top 10 features:")
print(imp.sort_values("importance", ascending=False).head(10).to_string(index=False))

  saved fig10_feature_importance.png
Top 10 features:
            feature  importance
         prev_delay    0.466878
       prev_delay_2    0.322269
    stop_hist_delay    0.053989
   route_hist_delay    0.029939
        delay_trend    0.025065
         agency_idx    0.014068
           stop_lon    0.013837
      scheduled_gap    0.013635
      stop_sequence    0.012197
mins_since_midnight    0.012006


## 7. Figure 11 — Predicted against actual

In [20]:
samp = (cv_preds.select("label", "prediction")
        .sample(fraction=min(1.0, 5000 / max(cv_preds.count(), 1)), seed=42)
        .toPandas())

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5))

a1.scatter(samp["label"], samp["prediction"], s=6, alpha=0.25, color="#3b7dd8")
lims = [samp[["label", "prediction"]].min().min(), samp[["label", "prediction"]].max().max()]
a1.plot(lims, lims, "r--", lw=1.4, label="Perfect prediction")
a1.set_xlabel("Actual delay (min)"); a1.set_ylabel("Predicted delay (min)")
a1.set_title("Predicted vs Actual"); a1.legend()

resid = samp["label"] - samp["prediction"]
a2.hist(resid, bins=60, color="#e76f51", edgecolor="none")
a2.axvline(0, color="black", ls="--", lw=1.2)
a2.set_xlabel("Residual (actual - predicted, min)"); a2.set_ylabel("Count")
a2.set_title(f"Residuals   mean={resid.mean():.3f}, sd={resid.std():.3f}")

save(fig, 11, "predicted_vs_actual")

  saved fig11_predicted_vs_actual.png


## 8. Figure 12 — Error by operator

In [21]:
by_op = (cv_preds
         .withColumn("abs_err", F.abs(F.col("label") - F.col("prediction")))
         .groupBy("agency_name")
         .agg(F.count("*").alias("n"),
              F.round(F.avg("abs_err"), 3).alias("mae"))
         .filter(F.col("n") >= 50)
         .orderBy("mae")
         .toPandas())

fig, ax = plt.subplots(figsize=(9, max(3.5, 0.34*len(by_op))))
ax.barh(by_op["agency_name"], by_op["mae"], color="#6a4c93")
ax.invert_yaxis()
ax.set_xlabel("Mean absolute error (minutes)")
ax.set_title("Prediction Error by Operator")
save(fig, 12, "error_by_operator")
by_op

  saved fig12_error_by_operator.png


,agency_name,n,mae
0,First Halifax,92,0.580
1,The Blackburn Bus Company,459,0.654
2,Bee Network,28181,0.721
3,Arriva North West,1640,0.852
4,High Peak,181,0.942
5,Warrington's Own Buses,771,1.013
6,Stagecoach Cumbria and North Lancashire,293,1.016
7,The Burnley Bus Company,130,1.022
8,Preston Bus,79,1.277
9,Huyton Travel,290,1.691


## 9. Persist the model

Saved so the dashboard or a later batch job can load it without retraining.

In [22]:
model_path = (MODELS / "rf_delay_model").as_posix()
prep_path  = (MODELS / "feature_pipeline").as_posix()

best.write().overwrite().save(model_path)
prep.write().overwrite().save(prep_path)

print(f"Model saved    : {model_path}")
print(f"Pipeline saved : {prep_path}")

comparison.to_csv(PROJECT / "docs" / "model_comparison.csv", index=False)
imp.sort_values("importance", ascending=False).to_csv(
    PROJECT / "docs" / "feature_importance.csv", index=False)
print("Result tables written to docs/")

Model saved    : E:/BODS-project/models/rf_delay_model
Pipeline saved : E:/BODS-project/models/feature_pipeline
Result tables written to docs/


In [23]:
print("Spark UI:", spark.sparkContext.uiWebUrl)

Spark UI: http://ujwal:4040


In [25]:
spark.stop()
print("Notebook 04 complete.")

Notebook 04 complete.
